In [1]:
import requests
import pandas as pd
import time
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

from src.config import load_config

CONFIG = load_config()

In [2]:
def get_meteo_data (lng: str,
                    lat: str,
					start_date: str,
					end_date: str) -> pd.DataFrame :
    
	print(f"Get: {lat}°N {lng}°E - {start_date}->{end_date}")

	params = {
		"latitude": lat,
		"longitude": lng,
		"start_date": start_date,
		"end_date": end_date,
		"daily": ["weather_code", "temperature_2m_max", "temperature_2m_min", "apparent_temperature_max", "apparent_temperature_min", "sunrise", "sunset", "daylight_duration", "sunshine_duration", "uv_index_max", "uv_index_clear_sky_max", "rain_sum", "showers_sum", "snowfall_sum", "precipitation_sum", "precipitation_hours", "precipitation_probability_max", "shortwave_radiation_sum", "et0_fao_evapotranspiration", "cloud_cover_mean", "dew_point_2m_mean", "et0_fao_evapotranspiration_sum", "relative_humidity_2m_mean", "snowfall_water_equivalent_sum", "pressure_msl_mean", "surface_pressure_mean", "visibility_mean", "wind_speed_10m_mean", "soil_moisture_0_to_100cm_mean", "soil_temperature_0_to_100cm_mean"],
	}

	try:
		r = requests.get(CONFIG["api"]["weather"]["url_archive"], params=params, timeout=(5, 30))
		r.raise_for_status()
		print (r.url)

	except requests.exceptions.Timeout:
		print("Timeout, nouvelle tentative...")
		time.sleep(2)
		
		r = requests.get(CONFIG["api"]["weather"]["url_archive"], params=params, timeout=(5, 30))
		r.raise_for_status()
		

	daily = r.json()["daily"]

	return pd.DataFrame({
		"date": pd.to_datetime(daily["time"]),
		"latitude": lat,
		"longitude": lng,
		"weather_code": daily["weather_code"],
		"temperature_2m_max": daily["temperature_2m_max"],
		"temperature_2m_min": daily["temperature_2m_min"],
		"apparent_temperature_max": daily["apparent_temperature_max"],
		"apparent_temperature_min": daily["apparent_temperature_min"],
		"sunrise": daily["sunrise"],
		"sunset": daily["sunset"],
		"daylight_duration": daily["daylight_duration"],
		"sunshine_duration": daily["sunshine_duration"],
		"uv_index_max": daily["uv_index_max"],
		"uv_index_clear_sky_max": daily["uv_index_clear_sky_max"],
		"rain_sum": daily["rain_sum"],
		"showers_sum": daily["showers_sum"],
		"snowfall_sum": daily["snowfall_sum"],
		"precipitation_sum": daily["precipitation_sum"],
		"precipitation_hours": daily["precipitation_hours"],
		"precipitation_probability_max": daily["precipitation_probability_max"],
		"shortwave_radiation_sum": daily["shortwave_radiation_sum"],
		"et0_fao_evapotranspiration": daily["et0_fao_evapotranspiration"],
		"cloud_cover_mean": daily["cloud_cover_mean"],
		"dew_point_2m_mean": daily["dew_point_2m_mean"],
		"et0_fao_evapotranspiration_sum": daily["et0_fao_evapotranspiration_sum"],
		"relative_humidity_2m_mean": daily["relative_humidity_2m_mean"],
		"snowfall_water_equivalent_sum": daily["snowfall_water_equivalent_sum"],
		"pressure_msl_mean": daily["pressure_msl_mean"],
		"surface_pressure_mean": daily["surface_pressure_mean"],
		"visibility_mean": daily["visibility_mean"],
		"wind_speed_10m_mean": daily["wind_speed_10m_mean"],
		"soil_moisture_0_to_100cm_mean": daily["soil_moisture_0_to_100cm_mean"],
		"soil_temperature_0_to_100cm_mean": daily["soil_temperature_0_to_100cm_mean"],
	})

In [3]:
import time

failed_calls = [] 

def fetch_meteo_by_year(code_bss: str,
                           lng: str,
                           lat: str,
                           end_date: str,
                           export_csv: bool = True) -> pd.DataFrame:
    
    start   = pd.to_datetime(CONFIG["api"]["weather"]["start_date"])
    end     = pd.to_datetime(end_date)
    frame   = []

    for year in range(start.year, end.year + 1):
        year_start = max(start, pd.Timestamp(year=year, month=1, day=1))
        year_end = min(end, pd.Timestamp(year=year, month=12, day=31))
        current_start = year_start.strftime("%Y-%m-%d")
        current_end = year_end.strftime("%Y-%m-%d")

        try:
            df_year = get_meteo_data(
                lng,
                lat,
                current_start,
                current_end
            )
            df_year["code_bss"] = code_bss

            if export_csv == True: 
                file = f"/home/ronanguilloueee/NappeCast/data/external/meteo_{code_bss.replace('/','')}_{pd.to_datetime(year_start).year}.csv"
                df_year.to_csv (file)

            frame.append(df_year)
            time.sleep(5)

        except Exception as e:
            print(f"Erreur station {code_bss}, année {year}: {e}")
            
            failed_calls.append({
                "code_bss": code_bss,
                "lng": lng,
                "lat": lat,
                "start_date": current_start,
                "end_date": current_end,
                "error": str(e),
            })

    if not frame:
        return pd.DataFrame()

    return pd.concat(frame, ignore_index=True)

In [5]:
file = f"/home/ronanguilloueee/NappeCast/data/external/station_data.csv"
df_station = pd.read_csv (file)

frame=[]
retry_frame = []

# récupération des data par années
for end_date, lat, lng, code_bss in zip(df_station["date_fin_mesure"], 
                                            df_station["latitude"], 
                                            df_station["longitude"], 
                                            df_station["code_bss"]):
    
    frame.append(fetch_meteo_by_year(code_bss,
                                       lng,
                                       lat,
                                       end_date,
                                       False))

df_meteo = pd.concat(frame, ignore_index=True)
file = f"/home/ronanguilloueee/NappeCast/data/external/meteo_data.csv"
df_meteo.to_csv (file)

Get: 2.685378283°N 42.685696389°E - 2017-01-01->2017-12-31
https://archive-api.open-meteo.com/v1/archive?latitude=2.685378283&longitude=42.685696389&start_date=2017-01-01&end_date=2017-12-31&daily=weather_code&daily=temperature_2m_max&daily=temperature_2m_min&daily=apparent_temperature_max&daily=apparent_temperature_min&daily=sunrise&daily=sunset&daily=daylight_duration&daily=sunshine_duration&daily=uv_index_max&daily=uv_index_clear_sky_max&daily=rain_sum&daily=showers_sum&daily=snowfall_sum&daily=precipitation_sum&daily=precipitation_hours&daily=precipitation_probability_max&daily=shortwave_radiation_sum&daily=et0_fao_evapotranspiration&daily=cloud_cover_mean&daily=dew_point_2m_mean&daily=et0_fao_evapotranspiration_sum&daily=relative_humidity_2m_mean&daily=snowfall_water_equivalent_sum&daily=pressure_msl_mean&daily=surface_pressure_mean&daily=visibility_mean&daily=wind_speed_10m_mean&daily=soil_moisture_0_to_100cm_mean&daily=soil_temperature_0_to_100cm_mean
Get: 2.685378283°N 42.68569

In [ ]:
# retraitement des erreurs
for call in failed_calls:
    try:
        df_retry = get_meteo_data(call["lng"], 
                                  call["lat"], 
                                  call["start_date"], 
                                  call["end_date"])
        
        df_retry["code_bss"] = call["code_bss"]
        
        file = f"/home/ronanguilloueee/NappeCast/data/external/meteo_{code_bss.replace('/','')}_{pd.to_datetime(call["start_date"]).year}.csv"
        df_retry.to_csv (file)
        
        retry_frame.append(df_retry)
        time.sleep(5)
    
    except Exception as e:
        print(f"Échec persistant pour {call['code_bss']} ({call['start_date']} - {call['end_date']}): {e}")

Get: 2.685378283°N 42.685696389°E - 2023-01-01->2023-12-31
Timeout, nouvelle tentative...
Get: 2.685378283°N 42.685696389°E - 2024-01-01->2024-12-31
Timeout, nouvelle tentative...
Get: 2.685378283°N 42.685696389°E - 2025-01-01->2025-12-31
Timeout, nouvelle tentative...
Get: 2.685378283°N 42.685696389°E - 2026-01-01->2026-07-13
https://archive-api.open-meteo.com/v1/archive?latitude=2.685378283&longitude=42.685696389&start_date=2026-01-01&end_date=2026-07-13&daily=weather_code&daily=temperature_2m_max&daily=temperature_2m_min&daily=apparent_temperature_max&daily=apparent_temperature_min&daily=sunrise&daily=sunset&daily=daylight_duration&daily=sunshine_duration&daily=uv_index_max&daily=uv_index_clear_sky_max&daily=rain_sum&daily=showers_sum&daily=snowfall_sum&daily=precipitation_sum&daily=precipitation_hours&daily=precipitation_probability_max&daily=shortwave_radiation_sum&daily=et0_fao_evapotranspiration&daily=cloud_cover_mean&daily=dew_point_2m_mean&daily=et0_fao_evapotranspiration_sum&

In [ ]:
df_retry = get_meteo_data(call["lng"], 
                                  call["lat"], 
                                  call["start_date"], 
                                  call["end_date"])
        
df_retry["code_bss"] = call["code_bss"]
   
        
file = f"/home/ronanguilloueee/NappeCast/data/external/meteo_{code_bss.replace('/','')}_{pd.to_datetime(call["start_date"]).year}.csv"
df_retry.to_csv (file)